In [ ]:
import os
import pandas as pd
import pickle
from prepare_sent_data import tokenize_sentence
import regex as re
from num2words import num2words
from helpers import load_config

def tokenize_sentences(language): 
    """
    Takes the raw language corpus and converts each sentence into a list of words
    E.g. 
    Raw data: Jamais je n'épouserai cet hérétique, vous m'entendez?
    Output data: ['jamais', 'je', 'ne', 'épouserai', 'cet', 'hérétique', 'vous', 'me', 'entendez']
    
    Takes: language key 
    Writes the data to a pickle file 
    """
    # Load configuration
    path = load_config["Sentence Data"]
    num2word_code = load_config["num2words Code"]

    # Prepare output directory
    output_dir = f"produced_data/{language}"
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"{language}_original_sentences.pkl")

    # Read and process sentences
    tokenized_sentences = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                sentence = line.strip()

                # Remove leading character if it's not a Unicode letter
                if sentence and not re.match(r'\p{L}', sentence[0]):
                    sentence = sentence[1:].lstrip()

                # convert numbers to words
                try:
                    sentence = re.sub(r'\d+', lambda x: num2words(int(x.group()), lang=num2word_code.lower()), sentence)
                except NotImplementedError:
                    print(f"Language '{language}' not supported by num2words. Skipping number conversion.")

                if not re.search(r'\p{L}', sentence, re.UNICODE):  # checks for any alphabetic Unicode letter
                    continue
                if any(char in sentence for char in ['(', ')', ':', '...']):
                    continue
                if sentence:
                    print(sentence)
                    tokenized = tokenize_sentence(sentence, language)
                    print(tokenized)
                    if tokenized:
                        tokenized_sentences.append(tokenized)
                
                # Optional: limit for testing
                if i >= 2000:
                    break
               
    except FileNotFoundError:
        print(f"Sentence data file not found: {path}")
        return

    # Save tokenized sentences
    try:
        with open(output_file, "wb") as f:
            pickle.dump(tokenized_sentences, f)
        print(f"✅ Saved tokenized sentences to '{output_file}'")
    except Exception as e:
        print(f"Failed to save tokenized data: {e}")


languages = ['FRA', 'DEU', 'JPN', 'CMN', 'VIE', 'YUE', 'ENG']

tokenize_sentences(languages[0])


In [2]:
file_path = "Z:/data/ENG/en.tok"  

# Read the tokenized sentences
with open(file_path, "r", encoding="utf-8") as f:
    sentences = [line.strip().split() for line in f if line.strip()]

# Inspect the first few sentences
for i, sent in enumerate(sentences[:5]):
    print(f"Sentence {i+1}: {sent}")


Sentence 1: ['My', 'parents', 'would', 'repudiate', 'my', 'brother', 'if', 'they', 'ever', 'found', 'out', 'he', 'was', 'gay', '.']
Sentence 2: ['In', 'order', 'to', 'keep', 'his', 'original', 'idea', 'from', 'being', 'copied', ',', 'Henry', 'resorted', 'to', 'reticence', '.']
Sentence 3: ['Please', 'tell', 'us', 'where', 'there', 'is', 'a', 'grocery', 'store', '.']
Sentence 4: ['You', 'should', 'not', 'give', 'him', 'up', 'for', 'lost', '.']
Sentence 5: ['You', 'are', 'too', 'critical', 'of', 'others', "'", 'shortcomings', '.']


In [ ]:
import pickle
from collections import Counter, defaultdict
from pathlib import Path
from helpers import get_ipa_espeak, clean_ipa, load_config
import json


def count_word_freq(language):
    """
    Count word frequencies using espeak-ng IPA, save as JSON lookup dict.
    Allows lookup by word or by IPA.
    """
    # Set up paths
    base_dir = Path(f"produced_data/{language}")
    input_path = base_dir / f"{language}_original_sentences.pkl"
    lookup_path = base_dir / f"{language}_word_lookup.json"

    # Load espeak-ng language code
    espeak_code = load_config(language, key="espeak Code")


    # Load tokenized data
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")      
    
    with input_path.open("rb") as f:
        text = pickle.load(f)

    counter = Counter()
    lookup = defaultdict(list)

    for sentence in text:
        for word in sentence:
            # Convert to IPA clean
            word_ipa = get_ipa_espeak(word, espeak_code)
            # Clean IPA
            cleaned_ipa = clean_ipa(word_ipa, True, '', language)
            if cleaned_ipa:
                # Count the word's frequency in the corpus
                counter[(word, cleaned_ipa)] += 1

    # Build lookup
    for (word, cleaned_ipa), freq in counter.items():
        lookup[word].append({"ipa": cleaned_ipa, "freq": freq})
        lookup[cleaned_ipa].append({"word": word, "freq": freq})

    # Save lookup dict as JSON
    with lookup_path.open("w", encoding="utf-8") as f:
        json.dump(dict(lookup), f, ensure_ascii=False, indent=2)

    print(f"✅ Lookup dictionary for frquency counts saved to: {lookup_path}")


count_word_freq('FRA')


✅ Lookup dictionary for frquency counts saved to: produced_data\FRA\FRA_word_lookup.json


In [ ]:
import subprocess
import numpy as np
import jieba
import pycantonese
import pandas as pd
from pypinyin import Style, pinyin as pypinyin_fn

def get_word_data(path, processing_type="sylls", clean=True):
    """
    Reads a CSV/TSV/Excel file of phonetically transcribed words and frequencies.
    Splits the words into tokens (syllables, phones, or characters), repeated according to frequency.

    Args:
        path (str): Path to the data file.
        processing_type (str): One of 'sylls', 'phones', 'chars'.
        clean (bool): Whether to clean IPA before tokenizing.

    Returns:
        list[list[str]]: Tokenized words, repeated by frequency.
    """

    # Extract language from filename robustly
    language = Path(path).parent.name.upper()
    print(f"Language: {language}")

    # Configuration for each language
    config = {
        "FRA": {
            "columns": ["syll", "freqfilms2"],
            "excel_col_filter": "freqfilms2",
            "delimiters": {"sylls": r"[.-]", "phonemes": "-"}
        },
        "DEU": {
            "columns": ["PhonStrsDISC", "Word Mann"],
            "excel_col_filter": "Word Mann",
            "delimiters": {"sylls": "-", "phonemes": "-"}
        },
        "ENG": {"delimiters": {"sylls": r"[-.]", "phonemes": "-"}},
        "CMN": {"delimiters": {"sylls": "_", "phonemes": "-"}},
        "VIE": {"delimiters": {"sylls": "_", "phonemes": "-"}},
        "JPN": {"delimiters": {"sylls": "_", "phonemes": "-"}},
        "YUE": {"delimiters": {"sylls": "_", "phonemes": "-"}},
    }

    if language not in config:
        raise ValueError(f"Unsupported language: {language}")

    lang_cfg = config[language]
    delimiter = lang_cfg.get("delimiters", {}).get(processing_type, "_")

    words = []

    if language in ["FRA", "DEU"]:
        # Load appropriate file format
        if path.endswith(".xlsx"):
            df = pd.read_excel(path, engine="openpyxl")
            df = df[df[lang_cfg["excel_col_filter"]] > 0]
        else:
            df = pd.read_csv(path, sep="\t", encoding="utf-8")
        for _, row in df.iterrows():
            raw_word = str(row[lang_cfg["columns"][0]])
            freq = int(row[lang_cfg["columns"][1]])
            tokens = tokenize(raw_word, processing_type, delimiter, clean, language)

            words.extend([tokens] * freq)

    else:  # Text-based format (e.g. ENG, CMN, VIE, JPN, YUE)
        with open(path, 'r', encoding="utf-8") as file:
            for line in file:
                try:
                    raw_word, freq = line.strip().split('\t')
                    freq = int(freq)
                    tokens = tokenize(raw_word, processing_type, delimiter, clean, language)
                    words.extend([tokens] * int(float(freq)))
                except ValueError:
                    continue  # skip malformed lines

    return words


# --- Mandarin ---
def cmn_to_ipa(text):
    words = jieba.lcut(text)
    ipa_words_list = []
    for word in words:
        pinyins = pypinyin_fn(word, style=Style.TONE3, heteronym=False)
        syllables = [syll[0] for syll in pinyins]
        ipa_words_list.append("_".join(syllables))
    return ipa_words_list


# --- Cantonese ---

def yue_to_ipa(text):
    jyutping_list = pycantonese.characters_to_jyutping(text)
    ipa_words_dict = [jp for jp in jyutping_list if jp]
    ipa_words_list = [word[1] for word in ipa_words_dict if word is not None and word[1] is not None]
    return ipa_words_list

def text_to_ipa(language):
    language_code_dict = {
        'cat': 'ca', 'cmn': 'zh', 'deu': 'de', 'eng': 'en', 'eus': 'eu',
        'fin': 'fi', 'fra': 'fr', 'hun': 'hu', 'ita': 'it', 'jpn': 'ja',
        'kor': 'ko', 'spa': 'es', 'srp': 'sr', 'tha': 'th', 'tur': 'tr',
        'vie': 'vi', 'yue': 'zh-yue'
    }

    # Load configuration CSV
    config_df = pd.read_json("C:/Users/emill/Documents/GitHub/Coupe_Expansion/emillys_code/language_config.json")
    config_df.set_index("Language", inplace=True) 

    try:
        lang_cfg = config_df.loc[language]
        espeak_lang = lang_cfg["espeak Code"]
    except KeyError:
        print(f"Language '{language}' not found in configuration.")
        return

    lang = language_key.lower()

    with open(path, "rb") as f:
        text = pickle.load(f)

    for sentence in text:
        if lang == "vie":
            return np.nan

        elif lang == "jpn":
            return np.nan

        elif lang == "tha":
            return np.nan

        elif lang == "cmn":
            print(text)
            print(f"IPA for {lang}: {cmn_to_ipa(text)}")
            return cmn_to_ipa(text)

        elif lang == "yue":
            print(text)
            print(f"IPA for {lang}: {yue_to_ipa(text)}")
            return yue_to_ipa(text)

        elif lang == "kor":
            return np.nan

        else:
            result = subprocess.run(
                ['espeak', '-q', '--ipa3', '-v', espeak_lang, text],
                capture_output=True,
                text=True
            )
            ipa_text = result.stdout.strip().replace('\n', ' ')
            print(ipa_text)
            print(f"IPA for {lang}: {ipa}")
            return ipa.split()

    # --- Apply to CSV ---
    df = pd.read_csv('semantically_similar_texts/semantically_similar_texts_with_ipa.csv')
    df['ipa'] = df.apply(lambda row: text_to_ipa(row['text'], row['language']), axis=1)
    df.to_csv('semantically_similar_texts/semantically_similar_texts_with_ipa.csv', sep='\t', index=False)